# Adapter

Ein linearer Adapter (d×d, Identitaet als Start) auf den eingefrorenen
Embeddings, Triplet-Loss mit Cosinus-Abstand, Hard Negatives aus dem
25–100-m-Ring. Trainiert wird ausschliesslich auf dem fit-Teil von `train`,
gestoppt auf Recall@1 eines Mini-Retrievals im val-Teil; database und query
werden hier nie beruehrt. Die Bausteine stehen in `src/adapter_training.py`.

Ergebnis: `weights/adapter/<method>_linear.pt` und die adaptierten
Embeddings `<method>_linear_embeddings.npy`, beide mit Fingerabdruck. Der
Adapter ist eine Vergleichszeile, kein Beitrag — er verliert gegen
Whitening (siehe README).

In [ ]:
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.adapter_training import (
    TripletDataset,
    apply_adapter,
    make_loader,
    split_fit_val,
    train_adapter,
    val_halves,
    val_recall_at_1,
)
from src.config import load_config, paths
from src.device import pick_device
from src.models.adapter import LinearAdapter
from src.pairs import train_positive_pairs
from src.run_guard import (
    adapter_fingerprint,
    embedding_fingerprint,
    print_run_header,
    require_fingerprint,
    validate_config,
    write_fingerprint,
)

CFG = load_config(PROJECT_ROOT)
PATHS = paths(CFG, PROJECT_ROOT)
validate_config(CFG)
VPR, TRAINING = CFG["vpr"], CFG["vpr"]["adapter_training"]
METHOD = VPR["method"]
SEED = int(VPR["split_seed"])
DEVICE = pick_device()
ADAPTER_KIND = "linear"                       # Gewichtsdatei und Embedding-Name

EMBEDDING_DIR = PATHS.embedding_dir(METHOD)
ADAPTER_PATH = PATHS.adapter_file(METHOD, ADAPTER_KIND)
ADAPTER_PATH.parent.mkdir(parents=True, exist_ok=True)
ADAPTED_NAME = f"{METHOD}_{ADAPTER_KIND}"

np.random.seed(SEED)
torch.manual_seed(SEED)

print_run_header(CFG, "05_adapter", device=DEVICE)
print(f"Epochen {TRAINING['epochs']}, lr {TRAINING['learning_rate']}, Marge {TRAINING['margin']}, "
      f"Batch {TRAINING['batch_size']}, Hard-Negative-Anteil {TRAINING['hard_negative_probability']}")

## Basis-Embeddings

Bewusst die Baseline ohne Adapter-Suffix: der Adapter wird auf dem
unveraenderten Encoder-Ausgang trainiert, sonst stapelt man ihn auf sich
selbst.

In [ ]:
embedding_path = EMBEDDING_DIR / f"{METHOD}_embeddings.npy"
embedding_metadata = pd.read_parquet(EMBEDDING_DIR / f"{METHOD}_metadata.parquet")
require_fingerprint(embedding_path,
                    embedding_fingerprint(CFG, METHOD, "none", embedding_metadata),
                    what="Basis-Embeddings")
embeddings = np.load(embedding_path, mmap_mode="r")
assert len(embeddings) == len(embedding_metadata)
print(f"Embeddings: {embeddings.shape}")

## fit / val innerhalb von train, Triplets

Sequenzen sind die Einheit: Frames derselben Fahrt liegen nie in fit und
val zugleich. Die val-Sequenzen werden halbiert in Mini-Database und
Mini-Query; darauf laeuft nach jeder Epoche dieselbe Recall-Rechnung wie in
07 (`src/evaluation.py`). Paare kommen aus `src/pairs.py`, Paare mit einem
Partner in val fallen weg.

In [ ]:
fit_mask, val_mask = split_fit_val(embedding_metadata, float(VPR["val_fraction"]), SEED)
fit_metadata = embedding_metadata[fit_mask].reset_index(drop=True)
val_metadata = embedding_metadata[val_mask].reset_index(drop=True)
fit_embeddings = np.ascontiguousarray(embeddings[np.flatnonzero(fit_mask)])
val_embeddings = np.ascontiguousarray(embeddings[np.flatnonzero(val_mask)])
print(f"fit: {fit_mask.sum():,} Bilder   val: {val_mask.sum():,} Bilder")

db_mask, q_mask = val_halves(val_metadata)
val_db_emb, val_q_emb = val_embeddings[db_mask], val_embeddings[q_mask]
val_db_meta = val_metadata[db_mask].reset_index(drop=True)
val_q_meta = val_metadata[q_mask].reset_index(drop=True)
VAL_RADIUS_M = float(VPR["val_radius_m"])

paare = train_positive_pairs(embedding_metadata, float(VPR["positive_radius_m"]),
                             float(VPR["max_heading_diff_deg"]))
fit_index = pd.Series(np.arange(len(fit_metadata)), index=fit_metadata["image_id"].to_numpy())
a = fit_index.reindex(paare["anchor_image_id"].to_numpy()).to_numpy()
p = fit_index.reindex(paare["positive_image_id"].to_numpy()).to_numpy()
ok = ~np.isnan(a) & ~np.isnan(p)
positive_pairs = list(zip(a[ok].astype(int), p[ok].astype(int), paare["distance_m"].to_numpy()[ok]))
if not positive_pairs:
    raise RuntimeError("Keine Positive-Paare im fit-Split.")
print(f"Positive Paare (fit): {len(positive_pairs):,}   verworfen (val): {(~ok).sum():,}")

dataset = TripletDataset(
    fit_embeddings, positive_pairs, fit_metadata,
    positive_radius_m=float(VPR["positive_radius_m"]),
    uncertain_radius_m=float(VPR["uncertain_radius_m"]),
    hard_negative_min_m=float(VPR["hard_negative_min_m"]),
    hard_negative_max_m=float(VPR["hard_negative_max_m"]),
    hard_negative_probability=float(TRAINING["hard_negative_probability"]),
)
train_loader = make_loader(dataset, int(TRAINING["batch_size"]), TRAINING.get("max_pairs_per_epoch"))
print(f"Hard Negatives je Anchor (Mittel): {dataset.mean_hard:.1f}   "
      f"Batches je Epoche: {len(train_loader):,}")

## Training

Early Stopping auf val-Recall@1; gespeichert wird die beste Epoche.

In [ ]:
adapter = LinearAdapter(embedding_dim=embeddings.shape[1]).to(DEVICE)

def val_fn(model):
    return val_recall_at_1(model, val_db_emb, val_q_emb, val_db_meta, val_q_meta,
                           CFG, VAL_RADIUS_M, DEVICE)

best_state, best_epoch, best_recall, n_val = train_adapter(
    adapter, train_loader, val_fn,
    epochs=int(TRAINING["epochs"]), learning_rate=float(TRAINING["learning_rate"]),
    margin=float(TRAINING["margin"]), device=DEVICE,
    patience=TRAINING.get("patience"), min_delta=float(TRAINING.get("min_delta", 0.0)),
)
adapter.load_state_dict(best_state)
print(f"\nBestes Modell aus Epoche {best_epoch}: val Recall@1 = {best_recall:.4f} "
      f"({n_val:,} val-Queries mit Abdeckung)")

## Speichern

Adapter-Gewichte und die adaptierten Embeddings, beide mit Fingerabdruck.
Die Embeddings entstehen blockweise aus der Baseline — die Bilder muessen
nicht erneut durch das Modell.

In [ ]:
torch.save(adapter.state_dict(), ADAPTER_PATH)
write_fingerprint(
    ADAPTER_PATH,
    adapter_fingerprint(CFG, METHOD, embedding_fingerprint(CFG, METHOD, "none", embedding_metadata)),
    best_epoch=best_epoch, best_val_recall_at_1=best_recall, epochs_trained=int(TRAINING["epochs"]),
)

# Trainingsdaten freigeben -- bei MegaLoc sind das 7 GB.
del train_loader, dataset, fit_embeddings, val_embeddings
gc.collect()

adapted_path = EMBEDDING_DIR / f"{ADAPTED_NAME}_embeddings.npy"
kosinus = apply_adapter(adapter, embeddings, adapted_path, DEVICE)
embedding_metadata.to_parquet(EMBEDDING_DIR / f"{ADAPTED_NAME}_metadata.parquet", index=False)
write_fingerprint(
    adapted_path, embedding_fingerprint(CFG, METHOD, ADAPTER_KIND, embedding_metadata),
    derived_from=embedding_path.name, best_epoch=best_epoch,
)
print(f"Adapter:               {ADAPTER_PATH}")
print(f"Adaptierte Embeddings: {adapted_path}   (mittlerer Cosinus zur Baseline {kosinus:.4f})")